
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# 01 - Grouping and Aggregating Data

This demonstration will show how to perform grouping and aggregation operations using NYC Taxi trip data. We'll explore basic grouping, multiple aggregations, and window functions.

### Objectives
- Understand basic grouping operations in Spark
- Perform time-based analysis using aggregations
- Implement complex aggregations with multiple metrics
- Use window functions for advanced analytics
- Optimize aggregation performance

## REQUIRED - SELECT CLASSIC COMPUTE

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.

1. Wait a few minutes for the cluster to start.

1. Once the cluster is running, complete the steps above to select your cluster.

## A. Data Setup and Loading

First, let's load our taxi trip data and examine its structure.

In [0]:
from pyspark.sql.functions import *

# Read and displaying the taxi data
trips_df = spark.read.table("samples.nyctaxi.trips")

display(trips_df.limit(10))

tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,pickup_zip,dropoff_zip
2016-02-13T21:47:53Z,2016-02-13T21:57:15Z,1.4,8.0,10103,10110
2016-02-13T18:29:09Z,2016-02-13T18:37:23Z,1.31,7.5,10023,10023
2016-02-06T19:40:58Z,2016-02-06T19:52:32Z,1.8,9.5,10001,10018
2016-02-12T19:06:43Z,2016-02-12T19:20:54Z,2.3,11.5,10044,10111
2016-02-23T10:27:56Z,2016-02-23T10:58:33Z,2.6,18.5,10199,10022
2016-02-13T00:41:43Z,2016-02-13T00:46:52Z,1.4,6.5,10023,10069
2016-02-18T23:49:53Z,2016-02-19T00:12:53Z,10.4,31.0,11371,10003
2016-02-18T20:21:45Z,2016-02-18T20:38:23Z,10.15,28.5,11371,11201
2016-02-03T10:47:50Z,2016-02-03T11:07:06Z,3.27,15.0,10014,10023
2016-02-19T01:26:39Z,2016-02-19T01:40:01Z,4.42,15.0,10003,11222


## B. Basic Grouping Operations

Let's start with simple grouping operations to understand trip patterns by location.

In [0]:
# Count trips by pickup location, to show top 5 most popular pickup locations
location_counts = trips_df \
    .groupBy("pickup_zip") \
    .count() \
    .orderBy(desc("count"))

display(location_counts.limit(5))

pickup_zip,count
10001,1227
10003,1181
10011,1129
10021,1021
10018,1012


## C. Combining Multiple Aggregations

Let's perform multiple aggregations by location using the `agg()` method

In [0]:
# Perform multiple aggregations by location, order by most popular pickup locations
location_stats = trips_df \
    .groupBy("pickup_zip") \
    .agg(
        count("*").alias("total_trips"),
        round(avg("trip_distance"), 2).alias("avg_distance"),
        round(avg("fare_amount"), 2).alias("avg_fare"),
        round(sum("fare_amount"), 2).alias("total_fare_amt")
    ) \
    .orderBy(desc("total_trips"))

display(location_stats.limit(5))

pickup_zip,total_trips,avg_distance,avg_fare,total_fare_amt
10001,1227,2.21,10.62,13028.01
10003,1181,2.32,10.98,12965.5
10011,1129,2.29,10.91,12321.5
10021,1021,2.02,10.21,10424.0
10018,1012,2.6,11.4,11541.51


## D. Window Functions

Now let's use window functions for more advanced analytics.

In [0]:
from pyspark.sql.window import Window

# Create window specs for different ranking methods
window_by_trips = Window.orderBy(desc("total_trips"))
window_by_fare = Window.orderBy(desc("avg_fare"))

# Add different types of rankings
ranked_locations = location_stats \
    .withColumn("trips_rank", rank().over(window_by_trips)) \
    .withColumn("fare_rank", rank().over(window_by_fare)) \
    .withColumn("fare_quintile", ntile(5).over(window_by_fare))  # Divide into 5 groups by fare

In [0]:
ranked_locations.createOrReplaceTempView("ranked_locations")

In [0]:
%sql
select * from ranked_locations

pickup_zip,total_trips,avg_distance,avg_fare,total_fare_amt,trips_rank,fare_rank,fare_quintile
8876,1,0.0,260.0,260.0,101,1,1
7974,1,0.0,188.0,188.0,101,2,1
7114,1,0.0,105.0,105.0,101,3,1
7310,1,0.0,105.0,105.0,101,3,1
7311,1,2.0,60.0,60.0,101,5,1
11430,4,17.28,52.0,208.0,79,6,1
11368,1,8.9,52.0,52.0,101,6,1
11420,1,14.8,52.0,52.0,101,6,1
11422,429,15.5,44.78,19209.5,24,9,1
11436,11,12.96,42.32,465.5,73,10,1


In [0]:
%sql
select fare_quintile,min(avg_fare),max(avg_fare),count(*) as cnt_per_group from ranked_locations group by fare_quintile

fare_quintile,min(avg_fare),max(avg_fare),cnt_per_group
1,20.33,260.0,26
2,13.75,20.0,26
3,11.44,13.73,26
4,10.24,11.4,25
5,0.0,10.21,25


In [0]:
# Displaying the results
display(ranked_locations.select(
    "pickup_zip", 
    "total_trips", 
    "avg_fare", 
    "avg_distance",
    "trips_rank",
    "fare_rank",
    "fare_quintile"
))

pickup_zip,total_trips,avg_fare,avg_distance,trips_rank,fare_rank,fare_quintile
8876,1,260.0,0.0,101,1,1
7974,1,188.0,0.0,101,2,1
7114,1,105.0,0.0,101,3,1
7310,1,105.0,0.0,101,3,1
7311,1,60.0,2.0,101,5,1
11430,4,52.0,17.28,79,6,1
11368,1,52.0,8.9,101,6,1
11420,1,52.0,14.8,101,6,1
11422,429,44.78,15.5,24,9,1
11436,11,42.32,12.96,73,10,1


## Key Takeaways

1. **Basic Grouping**
   - Use `groupBy()` followed by aggregation method
   - Can group by multiple columns
   - Always check data distribution

2. **Window Functions**
   - Perfect for comparative analytics
   - Consider performance impact
   - Use appropriate window frame

3. **Best Practices**
   - Always alias aggregated columns
   - Handle null values appropriately
   - Consider data skew in grouping keys


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>
